# 📥 Sprint 1 — Collecte données SIRENE
**Auteur** : Lucie Pintiaux  
**Date** : 08/05/2026  
**Objectif** : Télécharger et filtrer les données SIRENE pour le Nord (59)

---

## 📋 User Stories
- **US-002** : Télécharger SIRENE StockEtablissement complet
- **US-003** : Filtrer DEP=59 + NAF=47xx  
- **US-004** : Documenter source et date extraction

## ⚠️ Info importante
- Fichier source : ~800 Mo compressé → ~4 Go décompressé
- Temps téléchargement estimé : 10-15 min
- Résultat attendu : 95 000 - 105 000 établissements

## 📥 US-002 — Téléchargement SIRENE StockEtablissement

In [1]:
import os
import requests
from pathlib import Path
from datetime import date
import time

# Chemins
project_root = Path.cwd().parent
data_raw = project_root / "data" / "raw"

# URL source officielle
SIRENE_URL = "https://files.data.gouv.fr/insee-sirene/StockEtablissement_utf8.zip"
today = date.today().strftime("%Y%m%d")
ZIP_PATH = data_raw / f"sirene_stock_{today}.zip"

print("=" * 55)
print("📥 CONFIGURATION TÉLÉCHARGEMENT SIRENE")
print("=" * 55)
print(f"📂 Dossier destination : {data_raw}")
print(f"🌐 URL source          : {SIRENE_URL}")
print(f"💾 Fichier ZIP         : {ZIP_PATH.name}")
print(f"📅 Date extraction     : {today}")
print("=" * 55)

📥 CONFIGURATION TÉLÉCHARGEMENT SIRENE
📂 Dossier destination : c:\Users\lpint\OneDrive\Bureau\Dynamique commerciale 59\dashboard-commercial-nord59\data\raw
🌐 URL source          : https://files.data.gouv.fr/insee-sirene/StockEtablissement_utf8.zip
💾 Fichier ZIP         : sirene_stock_20260507.zip
📅 Date extraction     : 20260507


## ⬇️ Fonction de téléchargement avec progression

In [2]:
def telecharger_fichier(url: str, destination: Path) -> bool:
    """
    Télécharge un fichier avec barre de progression.
    
    Args:
        url: URL du fichier à télécharger
        destination: Chemin de destination local
        
    Returns:
        True si succès, False sinon
    """
    # Si déjà téléchargé, on skip
    if destination.exists():
        taille_mb = destination.stat().st_size / (1024 * 1024)
        print(f"✅ Fichier déjà présent ({taille_mb:.0f} Mo) — téléchargement ignoré")
        return True
    
    print(f"⬇️  Démarrage téléchargement...")
    print(f"📁 Destination : {destination.name}")
    
    try:
        response = requests.get(url, stream=True, timeout=30)
        response.raise_for_status()
        
        # Taille totale
        taille_totale = int(response.headers.get("content-length", 0))
        taille_mb = taille_totale / (1024 * 1024)
        print(f"📦 Taille fichier : {taille_mb:.0f} Mo")
        
        # Téléchargement par chunks
        taille_telechargee = 0
        chunk_size = 1024 * 1024  # 1 Mo par chunk
        debut = time.time()
        
        with open(destination, "wb") as f:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
                    taille_telechargee += len(chunk)
                    progression = (taille_telechargee / taille_totale * 100) if taille_totale else 0
                    mo_telecharges = taille_telechargee / (1024 * 1024)
                    print(f"\r⏳ {progression:.1f}% — {mo_telecharges:.0f}/{taille_mb:.0f} Mo", end="")
        
        duree = time.time() - debut
        print(f"\n✅ Téléchargement terminé en {duree:.0f} secondes !")
        return True
        
    except requests.exceptions.RequestException as e:
        print(f"\n❌ Erreur téléchargement : {e}")
        if destination.exists():
            destination.unlink()
        return False

print("✅ Fonction téléchargement prête !")

✅ Fonction téléchargement prête !


## 🚀 Lancement du téléchargement

In [3]:
# Lancement du téléchargement
print("=" * 55)
print("🚀 LANCEMENT TÉLÉCHARGEMENT SIRENE")
print("=" * 55)

succes = telecharger_fichier(SIRENE_URL, ZIP_PATH)

if succes:
    taille_mb = ZIP_PATH.stat().st_size / (1024 * 1024)
    print(f"📦 Taille ZIP : {taille_mb:.0f} Mo")
    print("=" * 55)
    print("✅ US-002 : Téléchargement SIRENE — DONE")
else:
    print("❌ Échec du téléchargement — relancer la cellule")

🚀 LANCEMENT TÉLÉCHARGEMENT SIRENE
⬇️  Démarrage téléchargement...
📁 Destination : sirene_stock_20260507.zip

❌ Erreur téléchargement : 404 Client Error: Not Found for url: https://files.data.gouv.fr/insee-sirene/StockEtablissement_utf8.zip
❌ Échec du téléchargement — relancer la cellule


In [4]:
# Chercher la bonne URL sur data.gouv.fr
import requests

# Nouvelle URL possible
urls_a_tester = [
    "https://files.data.gouv.fr/insee-sirene/StockEtablissement_utf8.zip",
    "https://www.data.gouv.fr/fr/datasets/r/0651fb76-bcf3-4f6a-a38d-bc04fa708576",
    "https://files.data.gouv.fr/insee-sirene/StockEtablissementHistorique_utf8.zip",
]

for url in urls_a_tester:
    try:
        r = requests.head(url, timeout=10, allow_redirects=True)
        print(f"{r.status_code} — {url}")
    except Exception as e:
        print(f"❌ Erreur — {url} : {e}")

404 — https://files.data.gouv.fr/insee-sirene/StockEtablissement_utf8.zip
200 — https://www.data.gouv.fr/fr/datasets/r/0651fb76-bcf3-4f6a-a38d-bc04fa708576
404 — https://files.data.gouv.fr/insee-sirene/StockEtablissementHistorique_utf8.zip


In [6]:
# Mise à jour avec la nouvelle URL officielle
SIRENE_URL = "https://object.files.data.gouv.fr/data-pipeline-open/siren/stock/StockEtablissement_utf8.zip"

print("=" * 55)
print("🚀 LANCEMENT TÉLÉCHARGEMENT SIRENE (nouvelle URL)")
print("=" * 55)
print("⚠️  Fichier 2,6 Go — prévoir 15-20 min")
print("=" * 55)

succes = telecharger_fichier(SIRENE_URL, ZIP_PATH)

if succes:
    taille_mb = ZIP_PATH.stat().st_size / (1024 * 1024)
    print(f"📦 Taille ZIP : {taille_mb:.0f} Mo")
    print("=" * 55)
    print("✅ US-002 : Téléchargement SIRENE — DONE")
else:
    print("❌ Échec — vérifier connexion internet")

🚀 LANCEMENT TÉLÉCHARGEMENT SIRENE (nouvelle URL)
⚠️  Fichier 2,6 Go — prévoir 15-20 min
⬇️  Démarrage téléchargement...
📁 Destination : sirene_stock_20260507.zip
📦 Taille fichier : 2692 Mo
⏳ 100.0% — 2692/2692 Mo
✅ Téléchargement terminé en 1342 secondes !
📦 Taille ZIP : 2692 Mo
✅ US-002 : Téléchargement SIRENE — DONE


## 🔍 US-003 — Filtrage DEP=59 + NAF=47xx

In [7]:
import zipfile
import pandas as pd

CSV_PATH = data_raw / "StockEtablissement_utf8.csv"

print("=" * 55)
print("📂 EXTRACTION ZIP")
print("=" * 55)

if CSV_PATH.exists():
    print("✅ CSV déjà extrait — extraction ignorée")
else:
    print("⏳ Extraction en cours (peut prendre 2-3 min)...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(data_raw)
    print("✅ Extraction terminée !")

# Aperçu des premières lignes (lecture légère)
print("\n⏳ Lecture aperçu (100 premières lignes)...")
df_apercu = pd.read_csv(CSV_PATH, nrows=100, low_memory=False)

print(f"📊 Colonnes disponibles : {len(df_apercu.columns)}")
print(f"📋 Noms des colonnes :")
for col in df_apercu.columns:
    print(f"   - {col}")

📂 EXTRACTION ZIP
⏳ Extraction en cours (peut prendre 2-3 min)...
✅ Extraction terminée !

⏳ Lecture aperçu (100 premières lignes)...
📊 Colonnes disponibles : 54
📋 Noms des colonnes :
   - siren
   - nic
   - siret
   - statutDiffusionEtablissement
   - dateCreationEtablissement
   - trancheEffectifsEtablissement
   - anneeEffectifsEtablissement
   - activitePrincipaleRegistreMetiersEtablissement
   - dateDernierTraitementEtablissement
   - etablissementSiege
   - nombrePeriodesEtablissement
   - complementAdresseEtablissement
   - numeroVoieEtablissement
   - indiceRepetitionEtablissement
   - dernierNumeroVoieEtablissement
   - indiceRepetitionDernierNumeroVoieEtablissement
   - typeVoieEtablissement
   - libelleVoieEtablissement
   - codePostalEtablissement
   - libelleCommuneEtablissement
   - libelleCommuneEtrangerEtablissement
   - distributionSpecialeEtablissement
   - codeCommuneEtablissement
   - codeCedexEtablissement
   - libelleCedexEtablissement
   - codePaysEtrangerEtablis

## 🔍 Filtrage Nord (59) + Commerce (47xx)

In [8]:
from datetime import date

OUTPUT_PATH = data_raw / f"sirene_nord59_{date.today().strftime('%Y%m%d')}.csv"

print("=" * 55)
print("🔍 FILTRAGE DEP=59 + NAF=47xx")
print("=" * 55)
print("⏳ Lecture par chunks (fichier ~4 Go)...")
print("   Patience, ~5-10 minutes...\n")

chunk_size = 100_000
lignes_lues = 0
lignes_gardees = 0
chunks_filtres = []

for chunk in pd.read_csv(
    CSV_PATH,
    chunksize=chunk_size,
    dtype=str,
    low_memory=False
):
    lignes_lues += len(chunk)
    
    # Filtre DEP=59 : code commune commence par 59
    masque_dep = chunk["codeCommuneEtablissement"].str.startswith("59", na=False)
    
    # Filtre NAF=47xx : activité principale commence par 47
    masque_naf = chunk["activitePrincipaleEtablissement"].str.startswith("47", na=False)
    
    # Les deux filtres combinés
    chunk_filtre = chunk[masque_dep & masque_naf]
    lignes_gardees += len(chunk_filtre)
    
    if len(chunk_filtre) > 0:
        chunks_filtres.append(chunk_filtre)
    
    # Progression
    print(f"\r⏳ {lignes_lues:,} lignes lues — {lignes_gardees:,} gardées", end="")

print(f"\n\n✅ Lecture terminée !")
print(f"📊 Total lignes lues   : {lignes_lues:,}")
print(f"📊 Lignes après filtre : {lignes_gardees:,}")
print(f"📊 Taux sélection      : {lignes_gardees/lignes_lues*100:.2f}%")

# Vérification fourchette attendue
if 95_000 <= lignes_gardees <= 115_000:
    print(f"\n✅ Volume dans la fourchette attendue (95k-115k) !")
else:
    print(f"\n⚠️  Volume hors fourchette — à vérifier")

🔍 FILTRAGE DEP=59 + NAF=47xx
⏳ Lecture par chunks (fichier ~4 Go)...
   Patience, ~5-10 minutes...

⏳ 43,315,488 lignes lues — 98,369 gardées

✅ Lecture terminée !
📊 Total lignes lues   : 43,315,488
📊 Lignes après filtre : 98,369
📊 Taux sélection      : 0.23%

✅ Volume dans la fourchette attendue (95k-115k) !


In [9]:
print("=" * 55)
print("💾 SAUVEGARDE DATASET FILTRÉ")
print("=" * 55)

# Concaténer tous les chunks
df_nord59 = pd.concat(chunks_filtres, ignore_index=True)

# Sauvegarder
df_nord59.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

taille_mb = OUTPUT_PATH.stat().st_size / (1024 * 1024)

print(f"✅ Fichier sauvegardé : {OUTPUT_PATH.name}")
print(f"📦 Taille             : {taille_mb:.1f} Mo")
print(f"📊 Lignes             : {len(df_nord59):,}")
print(f"📊 Colonnes           : {len(df_nord59.columns)}")
print()

# Aperçu
print("👀 Aperçu des premières lignes :")
df_nord59[["siret", "codeCommuneEtablissement", 
           "activitePrincipaleEtablissement",
           "etatAdministratifEtablissement",
           "dateCreationEtablissement"]].head(5)

💾 SAUVEGARDE DATASET FILTRÉ
✅ Fichier sauvegardé : sirene_nord59_20260507.csv
📦 Taille             : 22.3 Mo
📊 Lignes             : 98,369
📊 Colonnes           : 54

👀 Aperçu des premières lignes :


,siret,codeCommuneEtablissement,activitePrincipaleEtablissement,etatAdministratifEtablissement,dateCreationEtablissement
0,04544099700018,59178,47.08,F,NaN
1,04544147400017,59178,47.10,F,NaN
2,04544317300013,59456,47.04,F,NaN
3,04544497300015,59178,47.04,F,NaN
4,04565081900027,59606,47.59A,F,1900-01-01


## 📄 US-004 — Documentation source et métadonnées

In [10]:
from datetime import datetime
import hashlib

# Calcul checksum MD5 du fichier filtré
print("⏳ Calcul checksum MD5...")
md5 = hashlib.md5()
with open(OUTPUT_PATH, "rb") as f:
    for chunk in iter(lambda: f.read(8192), b""):
        md5.update(chunk)
checksum = md5.hexdigest()

# Statistiques rapides
nb_actifs = (df_nord59["etatAdministratifEtablissement"] == "A").sum()
nb_fermes = (df_nord59["etatAdministratifEtablissement"] == "F").sum()
nb_communes = df_nord59["codeCommuneEtablissement"].nunique()

# Contenu METADATA.md
metadata_content = f"""# 📋 METADATA — Dataset SIRENE Nord 59

## Source
- **Producteur** : INSEE / data.gouv.fr
- **URL source** : https://object.files.data.gouv.fr/data-pipeline-open/siren/stock/StockEtablissement_utf8.zip
- **URL stable** : https://www.data.gouv.fr/api/1/datasets/r/0651fb76-bcf3-4f6a-a38d-bc04fa708576
- **Date mise à jour source** : 01/05/2026
- **Date extraction** : {datetime.now().strftime('%d/%m/%Y %H:%M')}

## Périmètre
- **Département** : Nord (59) — code commune commençant par 59
- **Secteur** : Commerce de détail (NAF 47xx)
- **Filtre** : codeCommuneEtablissement LIKE '59%' AND activitePrincipaleEtablissement LIKE '47%'

## Fichier produit
- **Nom** : {OUTPUT_PATH.name}
- **Taille** : {OUTPUT_PATH.stat().st_size / (1024*1024):.1f} Mo
- **Checksum MD5** : {checksum}
- **Encodage** : UTF-8

## Statistiques
- **Total établissements** : {len(df_nord59):,}
- **Établissements actifs (A)** : {nb_actifs:,}
- **Établissements fermés (F)** : {nb_fermes:,}
- **Communes couvertes** : {nb_communes}
- **Colonnes** : {len(df_nord59.columns)}

## Reproduction
```bash
# Télécharger le fichier source
wget https://object.files.data.gouv.fr/data-pipeline-open/siren/stock/StockEtablissement_utf8.zip

# Filtrer (voir notebook 01_sprint1_collecte_sirene.ipynb)
# DEP=59 : codeCommuneEtablissement LIKE '59%'
# NAF=47 : activitePrincipaleEtablissement LIKE '47%'
```

## Notes
- Le fichier source complet contient ~43 millions d'établissements (France entière)
- Le dataset filtré représente 0.23% du total national
- Certains établissements anciens ont des dates manquantes (nettoyage Sprint 2)
"""

metadata_path = data_raw / "METADATA.md"
with open(metadata_path, "w", encoding="utf-8") as f:
    f.write(metadata_content)

print("✅ METADATA.md créé !")
print()
print(f"📊 Actifs  : {nb_actifs:,}")
print(f"📊 Fermés  : {nb_fermes:,}")
print(f"📊 Communes: {nb_communes}")
print(f"🔐 MD5     : {checksum}")
print()
print("✅ US-004 : Documentation — DONE")

⏳ Calcul checksum MD5...
✅ METADATA.md créé !

📊 Actifs  : 39,261
📊 Fermés  : 59,108
📊 Communes: 647
🔐 MD5     : 746d682394d895ea9de2908bb3583e57

✅ US-004 : Documentation — DONE


## 🏁 Bilan Sprint 1

In [11]:
print("=" * 55)
print("🏁 BILAN SPRINT 1 — COLLECTE DONNÉES SIRENE")
print("=" * 55)
print(f"📅 Date        : {date.today().strftime('%d/%m/%Y')}")
print(f"👤 Auteur       : Lucie Pintiaux")
print()
print("✅ US-002 : Téléchargement SIRENE         → DONE")
print("✅ US-003 : Filtrage DEP=59 + NAF=47xx    → DONE")
print("✅ US-004 : Documentation METADATA.md     → DONE")
print()
print("📊 RÉSULTATS :")
print(f"   Total établissements : 98,369")
print(f"   Actifs               : 39,261 (39.9%)")
print(f"   Fermés               : 59,108 (60.1%)")
print(f"   Communes couvertes   : 647/648")
print(f"   Colonnes             : 54")
print()
print("📁 FICHIERS PRODUITS :")
print(f"   data/raw/sirene_nord59_20260507.csv (22.3 Mo)")
print(f"   data/raw/METADATA.md")
print()
print("⏭️  SPRINT 2 : Nettoyage & Enrichissement")
print("=" * 55)

🏁 BILAN SPRINT 1 — COLLECTE DONNÉES SIRENE
📅 Date        : 07/05/2026
👤 Auteur       : Lucie Pintiaux

✅ US-002 : Téléchargement SIRENE         → DONE
✅ US-003 : Filtrage DEP=59 + NAF=47xx    → DONE
✅ US-004 : Documentation METADATA.md     → DONE

📊 RÉSULTATS :
   Total établissements : 98,369
   Actifs               : 39,261 (39.9%)
   Fermés               : 59,108 (60.1%)
   Communes couvertes   : 647/648
   Colonnes             : 54

📁 FICHIERS PRODUITS :
   data/raw/sirene_nord59_20260507.csv (22.3 Mo)
   data/raw/METADATA.md

⏭️  SPRINT 2 : Nettoyage & Enrichissement
